# 5. The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

This notebook turns the function-calling pattern from notebook 04 into a reusable agent loop. The model can call `search`, inspect the result, search again, and stop when it has enough information.

## Learning goals

By the end, you should be able to:

- explain the roles of instructions, tools, and memory;
- process one Responses API response;
- run a bounded loop until no function calls remain;
- understand typo recovery and prompt steering;
- add scope and safety controls before using an agent in an application.

**Practice rule:** run the cells in order. The first cells prepare the client, index, tool, and instructions used by every later experiment.

## 1. Environment setup and imports

This notebook reuses the local project modules and the OpenAI Responses API. The cell is intentionally not executed during setup so you can select your own kernel and confirm that your API key is available.

In [1]:
import json
import time

from dotenv import load_dotenv
from openai import OpenAI

from ingestion import build_index, load_faq_data

load_dotenv()
openai_client = OpenAI()

documents = load_faq_data()
index = build_index(documents)
print(f"Loaded and indexed {len(documents)} documents")

Loaded and indexed 1401 documents


## 2. Bring forward `search` and `search_tool` from notebook 04

The agent does not receive Python source code. It receives a tool schema, while the application keeps the actual search function. Keeping this contract identical to notebook 04 makes the experiments reproducible.

In [2]:
def search(query):
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"},
    )


search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

## 3. Define the base developer instructions

Instructions establish the agent's role and encourage it to search, inspect results, and refine its query. They steer behavior, but the application still needs validation and safety limits.

In [3]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## 4. Implement `make_call(call)` for tool dispatch

The model returns JSON arguments. The application parses them, dispatches to `search`, serializes the result, and returns the exact `function_call_output` structure expected by the Responses API.

In [4]:
def make_call(call):
    arguments = json.loads(call.arguments)

    if call.name != "search":
        raise ValueError(f"Unknown tool: {call.name}")

    result = search(**arguments)
    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json.dumps(result, indent=2),
    }

## 5. Run one Responses API turn

First process one response manually. This makes the state transition visible before we hide the same steps inside a loop.

In [5]:
question = "I just discovered the course. Can I join it?"
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        messages.append(make_call(item))
        has_function_calls = True
    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

print("Needs another model turn:", has_function_calls)

function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
function_call: search {"query":"new student join course enrollment FAQ discovered course"}
function_call: search {"query":"course access after start can I join FAQ"}
Needs another model turn: True


## 6. Build the full agentic `while` loop

The model decides how many searches are needed. The application keeps sending updated history until the response contains no function calls.

In [6]:
iteration = 1

while True:
    print(f"iteration #{iteration}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )
    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            messages.append(make_call(item))
            has_function_calls = True
        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    iteration += 1
    if not has_function_calls:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, just make sure you submit your project while submissions are still open. You can also start learning and working through the materials right away.

If you want, I can also help you with:
- how to start the course,
- whether you can still get a certificate,
- or how homework/project submissions work.

Anything else you’d like to explore?


## 7. Refactor into `agent_loop(...)`

Keep message history local to the function so separate user questions do not accidentally share state.

In [7]:
def agent_loop(instructions, question, model="gpt-5.4-mini"):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    iteration = 1
    last_answer = None

    while True:
        print(f"iteration #{iteration}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )
        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                messages.append(make_call(item))
                has_function_calls = True
            elif item.type == "message":
                last_answer = item.content[0].text
                print("ASSISTANT:")
                print(last_answer)

        iteration += 1
        if not has_function_calls:
            return last_answer

## 8. Test case: typo recovery

The lexical index may miss `Olama`. Watch the tool calls and see whether the model improves the search query on a later iteration.

In [8]:
typo_answer = agent_loop(
    instructions,
    "How do I run Olama locally?",
)
print(typo_answer)

iteration #1...
function_call: search {"query":"Olama local run install start Ollama locally"}
function_call: search {"query":"Ollama local setup run models FAQ"}
function_call: search {"query":"run Ollama locally command ollama serve"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS**: download and install from https://ollama.com/download
   - **Windows**: download the `.msi` installer from the same page
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model if needed and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s running, you should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
  

## 9. Test case: course enrollment

Compare this normal course question with the typo experiment. The important observation is not only the final answer, but how many tool calls were needed.

In [9]:
enrollment_answer = agent_loop(
    instructions,
    "I just discovered the course. Can I still join it?",
)
print(enrollment_answer)

iteration #1...
function_call: search {"query":"join course late discovered course can I still join"}
function_call: search {"query":"enroll after course started late join course FAQ"}
function_call: search {"query":"course registration deadline late enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join.

You can start the course anytime, and the materials/videos are available. If you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions.

If you’re interested, I can also help you figure out the best way to start catching up quickly. Is there anything else you’d like to explore?
Yes — you can still join.

You can start the course anytime, and the materials/videos are available. If you want a certificate, though, you’ll need to submit your project while the course is still accepting submissions.

If you’re interested, I can also help you figure out the best way to start catching up quickly. Is there anything else yo

## 10. Encourage multiple searches

A model may stop after the first useful result. Strengthen the instructions to ask it to search, analyze the result, and search again when another query could improve coverage.

In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use keywords from the user question in the first search.

Make multiple searches. First perform a search, analyze the results,
and then perform more searches with new keywords when useful.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

multi_search_answer = agent_loop(
    instructions,
    "I just discovered the course. Can I join it?",
)
print(multi_search_answer)

iteration #1...
function_call: search {"query":"join course enrollment late discovered course can I join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course and start learning.

If your goal is a certificate, the key thing is that you need to submit your project while the course is still accepting submissions. You can study the materials in self-paced mode, but certification requires finishing with a live cohort.

If you want, I can also explain how the certificate/project/peer-review process works.
Yes — you can still join the course and start learning.

If your goal is a certificate, the key thing is that you need to submit your project while the course is still accepting submissions. You can study the materials in self-paced mode, but certification requires finishing with a live cohort.

If you want, I can also explain how the certificate/project/peer-review process works.


## 11. Add off-topic restriction rules

Instructions can steer scope, but they are a lightweight guardrail. A production application should also validate inputs, tool permissions, and final answers.

In [11]:
instructions = """
You're a course teaching assistant.
Answer only questions about the course or its logistics.

If you need course information, use the search function.
Use the FAQ database as the only source of facts.
If the FAQ does not contain an answer, say that you don't know.
Do not answer off-topic questions from general knowledge.

Make multiple searches when the first result is incomplete.
At the end, ask if there are other course areas the learner wants to explore.
""".strip()

## 12. Run the off-topic test

Ask about the Queen's Gambit and inspect whether the agent stays within the FAQ scope instead of answering from general knowledge.

In [12]:
off_topic_answer = agent_loop(
    instructions,
    "what's queen gambit?",
)
print(off_topic_answer)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I don’t know based on the course FAQ.

If you want, I can help with other course areas you’d like to explore.
I don’t know based on the course FAQ.

If you want, I can help with other course areas you’d like to explore.


## 13. Optional safety controls

The simple loop is useful for learning, but production code should bound execution. Add a maximum iteration count, elapsed-time cutoff, and structured debug output.

In [13]:
def safe_agent_loop(
    instructions,
    question,
    model="gpt-5.4-mini",
    max_iterations=5,
    timeout_seconds=120,
):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]
    started_at = time.monotonic()

    for iteration in range(1, max_iterations + 1):
        elapsed = time.monotonic() - started_at
        if elapsed >= timeout_seconds:
            raise TimeoutError("Agent exceeded its time limit")

        print(f"iteration #{iteration} elapsed={elapsed:.1f}s")
        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )
        messages.extend(response.output)
        has_function_calls = False
        last_answer = None

        for item in response.output:
            print("output type:", item.type)
            if item.type == "function_call":
                print("tool:", item.name, item.arguments)
                messages.append(make_call(item))
                has_function_calls = True
            elif item.type == "message":
                last_answer = item.content[0].text
                print("assistant:", last_answer)

        if not has_function_calls:
            return last_answer

    raise RuntimeError("Agent reached max_iterations without a final answer")

## 14. Final study checklist

Before moving to framework comparisons, explain these boundaries in your own words:

- The model decides whether to call a tool and what arguments to use.
- Python executes the tool and returns a `function_call_output`.
- `messages` is the memory replayed on every model request.
- `call_id` links each tool result to the requested function call.
- No function calls means the loop can stop.
- Maximum iterations and timeouts prevent runaway execution.
- Tool logs, token usage, latency, and errors make the agent observable.

The handwritten loop is the foundation that frameworks package. Continue with [06-toyaikit-vs-handwritten-loop-notes.md](06-toyaikit-vs-handwritten-loop-notes.md) to compare the manual implementation with ToyAIKit.